In [ ]:
import pandas as pd

# path constants
DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

def load_hr(path):
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    return df.dropna(subset=['datetime'])

def load_psychometric(path):
    df = pd.read_csv(path)
    df['Question Start Time'] = pd.to_datetime(df['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    df['Question Answer Time'] = pd.to_datetime(df['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
    return df.dropna(subset=['Question Start Time'])

hr_baseline = load_hr(f'{DATA}/hr.csv')
hr_01 = load_hr(f'{DATA}/hr_01.csv')
hr_02 = load_hr(f'{DATA}/hr_02.csv')
hr_03 = load_hr(f'{DATA}/hr_03.csv')

psychometric_01 = load_psychometric(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = load_psychometric(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = load_psychometric(f'{PSY}/Psychometric_Test_Results_03.csv')

def get_average_hr_and_times(questions, hr_data):
    q = questions.sort_values('Question Start Time')
    merged = pd.merge_asof(
        q[['Question Start Time', 'Question Answer Time']],
        hr_data[['datetime', 'heart_rate']].sort_values('datetime'),
        left_on='Question Start Time',
        right_on='datetime',
        direction='nearest'
    )
    avg_hr = merged['heart_rate'].mean()
    start = q['Question Start Time'].min().strftime('%H:%M:%S')
    end = questions['Question Answer Time'].max().strftime('%H:%M:%S')
    return avg_hr, start, end

baseline_avg_hr = hr_baseline['heart_rate'].mean()

print('Setup complete.')
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM')

In [ ]:
test_types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']
sessions = [(psychometric_01, hr_01), (psychometric_02, hr_02), (psychometric_03, hr_03)]
yn = lambda x: 'Yes' if x else 'No'

rows = []
for test_type in test_types:
    for i, (psych, hr) in enumerate(sessions, 1):
        qs = psych[psych['Type'] == test_type].copy()
        avg_hr, start, end = get_average_hr_and_times(qs, hr)
        rows.append((
            f'Test {i:02d}', test_type, start, end, avg_hr,
            yn(avg_hr > baseline_avg_hr), yn(avg_hr > 100)
        ))

results = pd.DataFrame(rows, columns=[
    'Test', 'Type', 'Start Time', 'End Time',
    'Average HR (BPM)', 'Anxiety (Individual)', 'Anxiety (General >100 BPM)'
])
print(f'Baseline Average HR: {baseline_avg_hr:.2f} BPM\n')
print(results.to_string(index=False))